# 🌾 Crop Recommendation System
### Exploratory Data Analysis & Model Development
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Greens_d')
print('Libraries loaded ✔')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/crop_data.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nClass distribution:\n{df["label"].value_counts()}')

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
FEATURES = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
for ax, feat in zip(axes.flatten(), FEATURES):
    ax.hist(df[feat], bins=30, color='#52b788', edgecolor='white', alpha=0.85)
    ax.set_title(feat, fontsize=11)
    ax.set_ylabel('Count')
axes.flatten()[-1].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
corr = df[FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Greens', linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Mean nutrient levels per crop
crop_means = df.groupby('label')[['N', 'P', 'K']].mean().sort_values('N', ascending=False)
crop_means.plot(kind='bar', figsize=(14, 5), color=['#1b4332','#52b788','#b7e4c7'])
plt.title('Mean NPK Levels per Crop', fontsize=13, fontweight='bold')
plt.xlabel('')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Nutrient')
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preprocessing

In [ ]:
X = df[FEATURES].values
le = LabelEncoder()
y = le.fit_transform(df['label'].values)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

## 4. Model Training with GridSearchCV

In [ ]:
# Random Forest
rf_params = {'n_estimators': [100, 200], 'max_depth': [None, 10, 20], 'min_samples_split': [2, 5]}
rf_grid = GridSearchCV(RandomForestClassifier(random_state=42), rf_params, cv=5, scoring='accuracy', n_jobs=-1)
rf_grid.fit(X_train, y_train)
print('RF Best:', rf_grid.best_params_)
print('RF Test Acc:', accuracy_score(y_test, rf_grid.best_estimator_.predict(X_test)))

In [ ]:
# SVM
svm_params = {'C': [1, 10], 'kernel': ['rbf', 'poly'], 'gamma': ['scale', 'auto']}
svm_grid = GridSearchCV(SVC(probability=True, random_state=42), svm_params, cv=5, scoring='accuracy', n_jobs=-1)
svm_grid.fit(X_train_sc, y_train)
print('SVM Best:', svm_grid.best_params_)
print('SVM Test Acc:', accuracy_score(y_test, svm_grid.best_estimator_.predict(X_test_sc)))

In [ ]:
# KNN
knn_params = {'n_neighbors': [3, 5, 7, 11], 'weights': ['uniform', 'distance'], 'metric': ['euclidean', 'manhattan']}
knn_grid = GridSearchCV(KNeighborsClassifier(), knn_params, cv=5, scoring='accuracy', n_jobs=-1)
knn_grid.fit(X_train_sc, y_train)
print('KNN Best:', knn_grid.best_params_)
print('KNN Test Acc:', accuracy_score(y_test, knn_grid.best_estimator_.predict(X_test_sc)))

## 5. Results & Feature Importance

In [ ]:
importances = pd.Series(
    rf_grid.best_estimator_.feature_importances_, index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot(kind='barh', ax=ax, color='#52b788')
ax.set_title('Random Forest — Feature Importances', fontsize=12, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Classification report
y_pred = rf_grid.best_estimator_.predict(X_test)
print(classification_report(y_test, y_pred, target_names=le.classes_))